# Boost-Ens — 2차 부스터, ABE 체크포인트 앙상블 변형

`Boost.ipynb` 의 복사본이다. 바뀐 것은 **ABE 입력 로짓을 체크포인트 여러 벌의 평균으로 바꾸는 손잡이**
(`ABE_ENSEMBLE`) 하나뿐이고, 분할·검증·지표·저장 규율은 전부 동일하다.

**왜** — 원본 실행에서 축을 분해해 보니 부족한 쪽은 tie 축이 아니라 승부축이었다:

| 구성 | 총 ll | tie축 | 승부축 |
|---|---|---|---|
| ABE 단독 (ckpt-3000, T) | 0.9973 | 0.5909 | 0.4064 |
| AT (ABE+TB) | 0.9817 | **0.5756** | 0.4062 |
| V4 `w11+게이트(전부)` (모델 11개) | 0.9812 | 0.5770 | **0.4041** |

TB 는 tie 축을 0.5909 → 0.5756 으로 −0.0153 내렸다(V4 의 게이트 4개짜리 0.5770 보다도 낫다).
그런데 **TB 는 2-class tie 전용이라 승부축 기여가 구조적으로 정확히 0** 이다 —
`AT 승부축 0.4062` 가 `ABE 단독 0.4064` 와 같은 게 그 증거다. 승부축만 인코더 한 대로 남아 있다.

체크포인트 로짓 평균은 **추가 학습 없이** 바로 그 축을 때린다(별도 진단, T 는 valid 재피팅):

| ABE 구성 | 총 ll | 승부축 | winAUC |
|---|---|---|---|
| ckpt-3000 (원본 채택) | 0.9973 | 0.4064 | 0.7503 |
| 2500·3000·3500 평균 | 0.9898 | 0.4009 | 0.7585 |
| **전 체크포인트 평균** | **0.9872** | **0.4001** | 0.7589 |

`'all'` 은 stage2 지표로 고른 조합이 아니므로 선택 편향이 없다. 반면 **TB 는 앙상블이 손해**라
(0.5785 → 0.5800) `TB_ENSEMBLE` 은 껐다 — 손잡이만 남겨 재현 가능하게 둔다.

---

세 전문가의 **로짓만** 받아 3-class 확률을 내는 결합기. 두 구성을 같은 조건으로 비교한다.

| arm | 입력 | 열 |
|---|---|---|
| **AT**  | ABE 로짓 + TB 로짓 | 3 + 2 = 5 |
| **ATR** | ABE 로짓 + TB 로짓 + Arith 로짓 | 3 + 2 + 9 = 14 |

각 arm 을 부스터 3종(LGBM·XGBoost·CatBoost)으로 학습 → 6개 구성.

**검증** — stage2 10,748(valid 5,000 + holdout 5,748)로 5-fold 교차검증.
fold 마다 20%를 평가로 떼고, 남은 80%에서 다시 **10%(전체 기준)를 ES 전용**으로 뗀다.
지표는 **5 fold 값의 평균 ± 표준편차**다 — 5개 모델의 예측을 평균한 앙상블 점수가 아니다.
fold 분할은 원본과 같은 `(SEED, Y)` 라서 **원본 OOF 와 행 단위로 짝지어 비교할 수 있다**(전용 셀 있음).

**배포** — CV 는 측정 전용이고, 배포용 모델은 마지막 셀에서 stage2 **전량**으로 다시 적합해 저장한다.
산출물 경로는 원본과 겹치지 않게 `model_ens/`, `boost_oof_ens.npz` 로 뺀다.

**로짓 계약 (중요)** — 전문가마다 출력 형상과 swap 거동이 다르다. 이걸 틀리면 조용히 오염된다:

| 전문가 | 형상 | swap (A↔B 교환) 시 |
|---|---|---|
| ABE | (N, 3) `[z_a, z_b, z_tie]` | `z_a ↔ z_b` 교환 |
| **TB** | **(N, 2)** `[z_notie, z_tie]` | **불변 — 교환·재정렬 금지** |
| Arith | (N, 9) 부스터 3종 × `[p_a, p_b, p_tie]` (log p) | 블록마다 앞 두 열 교환 |

TB 는 2-class 라 `[1,0,2]` 재정렬을 적용하면 죽거나(운이 좋으면) 값이 뒤섞인다.
또한 TB 는 빈 응답 행에서 구간 마스크가 전부 0 이 되는데, 배포본 `AttnPool` 이
`torch.nan_to_num(w)` 로 0 벡터를 만들어 유한값을 보장한다(`Boost/Train_TB_run.ipynb` cell 25).
이 노트북은 그 결과만 읽으므로 **로드 직후 NaN·형상·id 정렬을 전부 assert** 해 계약을 강제한다.

로짓 평균은 **확률이 아니라 로짓 공간에서** 낸다 — 체크포인트마다 온도가 다른데
확률 평균은 그 차이를 뭉개고, 어차피 뒤에서 T 를 valid 로 다시 피팅하므로 로짓 평균이 일관된다.


### 설정 / Setup

In [ ]:
import os, json, glob, time, shutil
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42
np.random.seed(SEED)

# 경로는 절대경로로 고정 — 노트북 CWD 에 따라 산출물이 새는 사고를 막는다
# absolute paths: the notebook CWD must not decide where artifacts come from / go
LLM       = Path('.')
ABE_ROOT  = LLM / 'V5/aws/outputs/best_checkpoints'
TB_ROOT   = LLM / 'V6/pod/outputs/best_checkpoints'
ARITH_NPZ = LLM / 'Arith/outputs/arith_experts.npz'
ORIGIN    = LLM / 'Arith/origin'          # V5/origin 과 동일한 동결 분할 (id 대조로 검증)
OUT       = LLM / 'Boost'
MODEL_DIR = OUT / 'model_ens'             # 원본(model/)을 덮어쓰지 않는다
OOF_PATH  = OUT / 'boost_oof_ens.npz'
BASE_OOF  = OUT / 'boost_oof.npz'         # 원본 실행 OOF — 같은 fold 라 짝지어 비교한다

# ── 손잡이 ──
CKPT_SELECT = 'valid'   # 'valid': 프로토콜 준수(선택은 valid, hold 무접촉) | 'hold': hold 기준
ARITH_MODE  = 'all'     # 'all': 부스터 3종 로짓 9열 그대로 | 'mean': logp 평균 3열
USE_SWAP    = True      # 학습 swap 증강 + 예측 swap-TTA (전 단계와 동일 규율)
N_SPLITS    = 5
INNER_FRAC  = 0.10      # ES 전용 비율 — stage2 전체 기준 10%

# ── 이 노트북의 변경점 ──
# None: 단일 최선 체크포인트(= 원본 동작) | 'all': 전 체크포인트 로짓 평균 | [1500, 3000]: 명시 목록
# 명시 목록을 stage2 점수로 고르면 선택 편향이 된다. 'all' 은 고른 게 아니라서 안전하다.
ABE_ENSEMBLE = 'all'
TB_ENSEMBLE  = None     # 실측상 TB 는 앙상블이 손해(tie축 0.5785 -> 0.5800). 손잡이만 남긴다.

for p in (ABE_ROOT, TB_ROOT, ARITH_NPZ, ORIGIN / 'valid.csv', ORIGIN / 'holdout.csv'):
    assert p.exists(), f'경로 없음: {p}'
assert ABE_ENSEMBLE is None or ABE_ENSEMBLE == 'all' or isinstance(ABE_ENSEMBLE, (list, tuple))
assert TB_ENSEMBLE is None or TB_ENSEMBLE == 'all' or isinstance(TB_ENSEMBLE, (list, tuple))
os.makedirs(MODEL_DIR, exist_ok=True)
print(f'ckpt 선택 {CKPT_SELECT} | arith {ARITH_MODE} | swap {USE_SWAP} | {N_SPLITS}-fold, ES {INNER_FRAC:.0%}')
print(f'앙상블   ABE {ABE_ENSEMBLE} | TB {TB_ENSEMBLE}')


### stage2 로드 — valid 5,000 + holdout 5,748 (이 순서가 전 산출물의 행 순서다)

In [ ]:
def labels_of(d):
    y = np.select([d['winner_model_a'] == 1, d['winner_model_b'] == 1, d['winner_tie'] == 1],
                  [0, 1, 2], default=-1).astype(np.int64)
    assert (y != -1).all(), 'invalid labels!'
    return y

va = pd.read_csv(ORIGIN / 'valid.csv')
ho = pd.read_csv(ORIGIN / 'holdout.csv')
assert len(va) == 5000 and len(ho) == 5748, f'분할 크기 {len(va)}/{len(ho)}'
IDS = np.concatenate([va['id'].to_numpy(), ho['id'].to_numpy()])
Y = np.concatenate([labels_of(va), labels_of(ho)])
HALF = len(va)                                   # 앞 5,000 = valid, 뒤 5,748 = holdout
assert len(np.unique(IDS)) == len(IDS), 'id 중복'
prior = np.bincount(Y, minlength=3) / len(Y)
print(f'stage2 {len(IDS):,}행 (valid {HALF:,} + hold {len(IDS) - HALF:,}) | '
      f'사전분포 a/b/tie {np.round(prior, 4)}')

### 전문가 체크포인트 선정 + 로짓 로드 (앙상블 지원)

각 체크포인트의 `logits_export.npz` 에서 valid TTA 로짓으로 T 를 피팅하고 valid/hold 를 함께 채점한다
(각 학습 노트북의 검증 셀과 같은 공식). **선택은 `CKPT_SELECT` 기준 한쪽으로만** 한다 —
hold 로 고르면 그 hold 가 곧 부스터 평가의 절반이라 낙관이 섞인다. 기본값 `valid` 가 §1.6 프로토콜이다.

`*_ENSEMBLE` 이 켜지면 **선정 대신 지정된 체크포인트들의 로짓을 평균**해서 쓴다.
평균한 로짓은 스케일이 달라지므로 **T 를 평균 로짓 위에서 다시 피팅**해 단일본과 같은 잣대로 채점한다.
단일본 표와 앙상블 값을 한 화면에 같이 찍어 이득을 바로 읽을 수 있게 한다.

T 는 여기서 **적용하지 않는다.** 트리는 한 열을 상수로 나눠도 분할 순서가 안 바뀌어 결과가 동일하고,
부스터가 직접 확률을 내므로 별도 보정이 필요 없다. T 는 선택 지표를 만드는 데만 쓴다.


In [ ]:
import torch
import torch.nn as nn

def fit_temperature(logits, labels):
    z = torch.tensor(logits, dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.long)
    logT = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = nn.functional.cross_entropy(z / logT.exp(), y)
        loss.backward()
        return loss
    opt.step(closure)
    return float(logT.exp().detach())

# stage2 로짓 한 벌 = [valid; hold] 를 TTA 평균해 쌓은 것. 로드 즉시 정렬 계약을 강제한다.
def load_one(path):
    z = np.load(path)
    L = np.vstack([(z['valid_fwd'] + z['valid_swp']) / 2,
                   (z['hold_fwd'] + z['hold_swp']) / 2]).astype(np.float64)
    zid = np.concatenate([z['valid_ids'], z['hold_ids']])
    zlab = np.concatenate([z['valid_labels'], z['hold_labels']])
    assert (zid == IDS).all(), f'id 정렬 불일치 — 행이 어긋난 채 학습된다: {path}'
    assert (zlab == Y).all(), f'라벨 불일치: {path}'
    return L

# 주어진 로짓 행렬을 valid 로 T 피팅해 valid/hold 를 채점. ABE = 3-class ll, TB = binary tie ll.
def score_logits(tag, L):
    v, h = L[:HALF], L[HALF:]
    vl, hl = Y[:HALF], Y[HALF:]
    if tag == 'abe':
        T = fit_temperature(v, vl)
        sm = lambda x: torch.softmax(torch.tensor(x / T, dtype=torch.float32), 1).numpy()
        return log_loss(vl, sm(v), labels=[0, 1, 2]), log_loss(hl, sm(h), labels=[0, 1, 2]), T
    T = fit_temperature(v, (vl == 2).astype(np.int64))
    sm = lambda x: torch.softmax(torch.tensor(x / T, dtype=torch.float32), 1).numpy()[:, 1]
    b = lambda p, l: log_loss((l == 2).astype(int), np.c_[1 - p, p], labels=[0, 1])
    return b(sm(v), vl), b(sm(h), hl), T

def pick_and_load(tag, root, n_class, ensemble=None):
    paths = {int(p.rsplit('-', 1)[1]): Path(p)
             for p in glob.glob(str(root / 'checkpoint-*'))}
    rows = []
    for s in sorted(paths):
        L = load_one(paths[s] / 'logits_export.npz')
        lv, lh, T = score_logits(tag, L)
        rows.append((s, lv, lh, T, L))
    key = 1 if CKPT_SELECT == 'valid' else 2
    best = min(rows, key=lambda r: r[key])

    if ensemble is None:
        steps = [best[0]]
    elif ensemble == 'all':
        steps = sorted(paths)
    else:
        steps = sorted(int(s) for s in ensemble)
        missing = [s for s in steps if s not in paths]
        assert not missing, f'{tag} 체크포인트 없음: {missing}'

    metric = '3-class ll' if tag == 'abe' else 'binary tie ll'
    print(f'[{tag.upper()}] {metric:<14} {"step":>6} {"valid":>8} {"hold":>8} {"T":>7}')
    for s, lv, lh, T, _ in rows:
        mark = '  <-- 단독최선' if s == best[0] else ''
        mark += '  [앙상블]' if len(steps) > 1 and s in steps else ''
        print(f'{"":>21} {s:>6} {lv:8.4f} {lh:8.4f} {T:7.4f}{mark}')

    if len(steps) == 1:
        L = dict((r[0], r[4]) for r in rows)[steps[0]]
    else:
        # 로짓 공간 평균 — 확률 평균은 체크포인트별 온도 차이를 뭉갠다
        L = np.mean([dict((r[0], r[4]) for r in rows)[s] for s in steps], axis=0)
    lv, lh, T = score_logits(tag, L)                       # 평균 로짓 위에서 T 재피팅
    if len(steps) > 1:
        print(f'{"":>21} {"앙상블":>6} {lv:8.4f} {lh:8.4f} {T:7.4f}  <-- 채택 ({len(steps)}벌 평균)')
        d_v, d_h = lv - best[1], lh - best[2]
        print(f'{"":>21} {"":>6} {d_v:+8.4f} {d_h:+8.4f}          단독최선 대비 (음수 = 이득)')

    # ── 계약 강제 ── 형상·유한성. id/라벨은 load_one 에서 이미 검사했다
    assert L.shape == (len(IDS), n_class), f'{tag} 형상 {L.shape} != {(len(IDS), n_class)}'
    assert np.isfinite(L).all(), f'{tag} 비유한값 {int((~np.isfinite(L)).sum())}개 (빈 응답 가드 확인)'
    return steps, T, L

ABE_STEPS, ABE_T, L_ABE = pick_and_load('abe', ABE_ROOT, 3, ABE_ENSEMBLE)
print()
TB_STEPS, TB_T, L_TB = pick_and_load('tb', TB_ROOT, 2, TB_ENSEMBLE)
assert L_TB.shape[1] == 2, 'TB 는 2-class 다 — [1,0,2] 재정렬 금지'


### Arith 로짓 로드 — 부스터 3종의 log p

In [ ]:
za = np.load(ARITH_NPZ)
aid = np.concatenate([za['ids_valid'], za['ids_hold']])
alab = np.concatenate([za['y_valid'], za['y_hold']])
assert (aid == IDS).all(), 'Arith id 정렬 불일치'
assert (alab == Y).all(), 'Arith 라벨 불일치'

ARITH_MODELS = ('lgbm', 'xgb', 'cat')
blocks = [np.vstack([za[f'logp_valid__{m}'], za[f'logp_hold__{m}']]).astype(np.float64)
          for m in ARITH_MODELS]
if ARITH_MODE == 'mean':
    L_ARI = np.mean(blocks, axis=0)                       # 3열 — logp 평균
    ARI_COLS = [f'ari_mean_{c}' for c in ('a', 'b', 'tie')]
else:
    L_ARI = np.hstack(blocks)                             # 9열 — 부스터별로 다 준다
    ARI_COLS = [f'ari_{m}_{c}' for m in ARITH_MODELS for c in ('a', 'b', 'tie')]
assert np.isfinite(L_ARI).all(), 'Arith 비유한값'
print(f'Arith {L_ARI.shape} ({ARITH_MODE}) | 열: {ARI_COLS}')

### 행렬 조립 + swap 매핑

swap 은 **열 치환 하나**로 표현한다 — 순열 배열이면 `perm[perm] == 항등` 을 직접 검사할 수 있어
부호반전·역수처럼 손으로 짠 변환보다 실수 여지가 없다. TB 두 열은 순열에서 제자리에 남는다.

In [ ]:
ABE_COLS = ['abe_z_a', 'abe_z_b', 'abe_z_tie']
TB_COLS = ['tb_z_notie', 'tb_z_tie']

X_AT = np.hstack([L_ABE, L_TB])                   # arm AT
X_ATR = np.hstack([L_ABE, L_TB, L_ARI])           # arm ATR
COLS_AT, COLS_ATR = ABE_COLS + TB_COLS, ABE_COLS + TB_COLS + ARI_COLS

# A<->B 교환 시의 열 순열. ABE/Arith 는 a<->b 교환, TB 는 불변(제자리).
def make_perm(cols):
    idx = {c: i for i, c in enumerate(cols)}
    perm = np.arange(len(cols))
    pairs = [('abe_z_a', 'abe_z_b')]
    if ARITH_MODE == 'mean':
        pairs.append(('ari_mean_a', 'ari_mean_b'))
    else:
        pairs += [(f'ari_{m}_a', f'ari_{m}_b') for m in ARITH_MODELS]
    for a, b in pairs:
        if a in idx and b in idx:
            perm[idx[a]], perm[idx[b]] = idx[b], idx[a]
    return perm

PERM = {'AT': make_perm(COLS_AT), 'ATR': make_perm(COLS_ATR)}
for k, p in PERM.items():
    cols = COLS_AT if k == 'AT' else COLS_ATR
    assert (p[p] == np.arange(len(p))).all(), f'{k} 순열이 involution 이 아니다'
    fixed = [c for i, c in enumerate(cols) if p[i] == i]
    assert all(c.startswith('tb_') or c.endswith('_tie') for c in fixed), f'{k} 고정열 이상: {fixed}'
print('swap 순열 involution 통과 | 고정열(=swap 불변):',
      [c for i, c in enumerate(COLS_ATR) if PERM['ATR'][i] == i])

def swap_y(y):
    return np.where(y == 0, 1, np.where(y == 1, 0, 2)).astype(y.dtype)

ARMS = {'AT': (X_AT, COLS_AT), 'ATR': (X_ATR, COLS_ATR)}
for k, (X, c) in ARMS.items():
    print(f'  arm {k:<4} {X.shape}  {c}')

### 지표 — 축 분해 (총 ll = tie축 + 승부축)

임의의 3-class 확률벡터에 성립하는 항등식이라 매 fold 마다 검산까지 한다.
게이트가 tie 축만 갈아끼우는 구조이므로, 부스터가 어느 축에서 벌었는지 보는 것이 핵심이다.

In [ ]:
EPS = 1e-15

def per_row_ll(p, y):
    p = np.clip(np.asarray(p, np.float64), EPS, 1.0)
    p = p / p.sum(1, keepdims=True)
    return -np.log(p[np.arange(len(y)), y])

def axis_report(p, y):
    p = np.clip(np.asarray(p, np.float64), EPS, 1.0)
    p = p / p.sum(1, keepdims=True)
    total = per_row_ll(p, y).mean()
    it = y == 2
    t = np.clip(p[:, 2], EPS, 1 - EPS)
    tie_axis = float(-np.where(it, np.log(t), np.log(1 - t)).mean())
    nz = ~it
    rA = np.clip(p[nz, 0] / (p[nz, 0] + p[nz, 1]), EPS, 1 - EPS)
    win_axis = float(-np.where(y[nz] == 0, np.log(rA), np.log(1 - rA)).sum() / len(y))
    assert abs(total - tie_axis - win_axis) < 1e-6, '축 분해 검산 실패'
    return {'총 ll': float(total), 'tie축': tie_axis, '승부축': win_axis,
            'tie AUC': float(roc_auc_score(it.astype(int), t)),
            '승부 AUC': float(roc_auc_score((y[nz] == 0).astype(int), rA)),
            'acc%': float((p.argmax(1) == y).mean() * 100)}

# Δ = mean(ll_a − ll_b) 의 95% CI. 양수면 b 가 좋다.
def paired_ci(ll_a, ll_b, n_boot=2000, seed=SEED):
    d = np.asarray(ll_a) - np.asarray(ll_b)
    rng = np.random.default_rng(seed)
    bs = d[rng.integers(0, len(d), (n_boot, len(d)))].mean(1)
    return float(d.mean()), float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))

def fmt_ci(m, lo, hi, thresh=0.0005):
    sig = '유의' if (lo > 0 or hi < 0) else '노이즈'
    return f'{m:+.4f} [{lo:+.4f},{hi:+.4f}] {sig}' + (' ★' if sig == '유의' and abs(m) >= thresh else '')

### 부스터 3종 + 5-fold 하네스

파라미터는 Arith/Train.ipynb 와 같은 계열이되 **입력이 5~14열뿐이므로 용량을 낮췄다**
(leaves 63 → 15, depth 8 → 4). 10,748행에 5열짜리 문제에 큰 트리를 쓰면 ES 만으로는 과적합을 못 막는다.

In [ ]:
import inspect
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

def make_lgbm():
    return lgb.LGBMClassifier(objective='multiclass', num_class=3, learning_rate=0.03,
                              n_estimators=3000, num_leaves=15, min_child_samples=60,
                              colsample_bytree=0.9, subsample=0.9, subsample_freq=1,
                              reg_lambda=1.0, random_state=SEED, n_jobs=-1, verbosity=-1)

def make_xgb():
    return xgb.XGBClassifier(objective='multi:softprob', num_class=3, learning_rate=0.03,
                             n_estimators=3000, max_depth=4, min_child_weight=20,
                             subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
                             tree_method='hist', eval_metric='mlogloss',
                             early_stopping_rounds=100, random_state=SEED, n_jobs=-1, verbosity=0)

def make_cat():
    return cb.CatBoostClassifier(loss_function='MultiClass', eval_metric='MultiClass',
                                 iterations=3000, learning_rate=0.03, depth=4,
                                 l2_leaf_reg=6.0, early_stopping_rounds=100,
                                 bootstrap_type='Bernoulli', subsample=0.9, border_count=254,
                                 random_seed=SEED, verbose=0, allow_writing_files=False)

FACTORY = {'lgbm': make_lgbm, 'xgb': make_xgb, 'cat': make_cat}
_LGB_NEW = 'eval_X' in inspect.signature(lgb.LGBMClassifier.fit).parameters

def fit_one(name, m, Xf, yf, X_es, y_es):
    if name == 'lgbm':
        ekw = {'eval_X': X_es, 'eval_y': y_es} if _LGB_NEW else {'eval_set': [(X_es, y_es)]}
        m.fit(Xf, yf, **ekw, callbacks=[lgb.early_stopping(100, verbose=False)])
    elif name == 'xgb':
        m.fit(Xf, yf, eval_set=[(X_es, y_es)], verbose=False)
    else:
        m.fit(Xf, yf, eval_set=(X_es, y_es), use_best_model=True)
    return m

def best_iter(name, m):
    if name == 'lgbm':
        return int(m.booster_.best_iteration or 0)
    if name == 'xgb':
        return int(getattr(m, 'best_iteration', 0) or 0)
    return int(m.get_best_iteration() or 0)

# swap-TTA — 교환 뷰의 예측은 클래스도 되돌려야 한다([1,0,2]). TB 열은 순열이 제자리라 안전.
def predict_tta(m, Xp, perm):
    p = m.predict_proba(Xp)
    if not USE_SWAP:
        return p
    return (p + m.predict_proba(Xp[:, perm])[:, [1, 0, 2]]) / 2

# fold 마다 평가 20% / ES 10%(전체 기준) / 나머지 학습. 지표는 fold 값 자체를 모은다.
def cv_boost(name, X, y, perm, n_splits=N_SPLITS, inner_frac=INNER_FRAC, seed=SEED):
    n_inner = int(round(inner_frac * len(y)))
    skf = StratifiedKFold(n_splits, shuffle=True, random_state=seed)
    oof = np.zeros((len(y), 3))
    rows, iters = [], []
    for k, (rest, te) in enumerate(skf.split(X, y)):
        tr, es = train_test_split(rest, test_size=n_inner, stratify=y[rest],
                                  random_state=seed + k, shuffle=True)
        if USE_SWAP:
            Xf = np.vstack([X[tr], X[tr][:, perm]])
            yf = np.hstack([y[tr], swap_y(y[tr])])
        else:
            Xf, yf = X[tr], y[tr]
        m = fit_one(name, FACTORY[name](), Xf, yf, X[es], y[es])
        p = predict_tta(m, X[te], perm)
        oof[te] = p
        rows.append(axis_report(p, y[te]))
        iters.append(best_iter(name, m))
    fold_df = pd.DataFrame(rows)
    return {'fold': fold_df, 'mean': fold_df.mean(), 'std': fold_df.std(ddof=1),
            'oof': oof, 'iters': iters}

### 실행 — 2 arm × 부스터 3종

In [ ]:
R = {}
for arm, (X, cols) in ARMS.items():
    for name in FACTORY:
        t0 = time.time()
        R[(arm, name)] = cv_boost(name, X, Y, PERM[arm])
        r = R[(arm, name)]
        print(f"{arm:<4} {name:<5} 총 ll {r['mean']['총 ll']:.4f} ± {r['std']['총 ll']:.4f}"
              f"   tie축 {r['mean']['tie축']:.4f}  승부축 {r['mean']['승부축']:.4f}"
              f"   iters {r['iters']}  ({time.time() - t0:.0f}s)")

### 결과 — fold 평균이 판정 지표

`blend-w11` 0.9844 / `w11+게이트(전부)` 0.9812 는 V4 가 같은 stage2 10,748 에서 낸 값이라 직접 비교된다.
단 그쪽은 5-fold OOF 전체 점수이고 여기는 fold 평균이라, 0.001 급 차이는 프로토콜 차이로도 난다.

In [ ]:
V4_BLEND, V4_GATE = 0.9844, 0.9812      # V4 test10 (같은 stage2 10,748)

tbl = pd.DataFrame({f'{a}·{n}': R[(a, n)]['mean'] for a, n in R}).T
tbl['ll std'] = [R[(a, n)]['std']['총 ll'] for a, n in R]
tbl['vs V4게이트'] = tbl['총 ll'] - V4_GATE
display(tbl.sort_values('총 ll').round(4))

print('\n[fold 별 총 ll]')
fold_tbl = pd.DataFrame({f'{a}·{n}': R[(a, n)]['fold']['총 ll'].values for a, n in R},
                        index=[f'fold{k}' for k in range(N_SPLITS)])
display(fold_tbl.round(4))

print('\n[기준선]', f'사전분포 {-np.log(prior[Y]).mean():.4f}', f'| 균등 {np.log(3):.4f}',
      f'| V4 blend {V4_BLEND} | V4 게이트 {V4_GATE}')

### arm 비교 — Arith 로짓을 넣으면 실제로 좋아지나

같은 fold·같은 행에서 나온 OOF 예측이라 paired bootstrap 이 성립한다.
이게 "산술 전문가가 앙상블에 기여하는가"에 대한 **배포 충실 판정**이다 —
모든 로짓이 stage2 에서 out-of-sample 이고 구조도 실제 파이프라인과 같다.

In [ ]:
print('=== AT -> ATR (양수 = Arith 로짓 추가가 이득, 문턱 0.0005) ===')
for name in FACTORY:
    a, b = R[('AT', name)]['oof'], R[('ATR', name)]['oof']
    print(f'  {name:<5} ' + fmt_ci(*paired_ci(per_row_ll(a, Y), per_row_ll(b, Y))))

print('\n=== 부스터 간 (arm 안에서, lgbm 기준) ===')
for arm in ARMS:
    base = per_row_ll(R[(arm, 'lgbm')]['oof'], Y)
    for name in ('xgb', 'cat'):
        print(f'  {arm:<4} lgbm -> {name:<5} '
              + fmt_ci(*paired_ci(base, per_row_ll(R[(arm, name)]['oof'], Y))))

print('\n=== 3종 logp 평균 앙상블 (arm 별) ===')
for arm in ARMS:
    lp = np.mean([np.log(np.clip(R[(arm, n)]['oof'], EPS, 1)) for n in FACTORY], axis=0)
    p_ens = np.exp(lp)
    p_ens /= p_ens.sum(1, keepdims=True)
    rep = axis_report(p_ens, Y)
    best_single = min(FACTORY, key=lambda n: log_loss(Y, R[(arm, n)]['oof'], labels=[0, 1, 2]))
    d = paired_ci(per_row_ll(R[(arm, best_single)]['oof'], Y), per_row_ll(p_ens, Y))
    print(f"  {arm:<4} 앙상블 총 ll {rep['총 ll']:.4f} (tie축 {rep['tie축']:.4f} / "
          f"승부축 {rep['승부축']:.4f})  vs 단독최선({best_single}) " + fmt_ci(*d))

print('\n[valid/hold 반쪽 분리 — 이득이 한쪽에만 있으면 선택 편향 신호]')
for arm in ARMS:
    for name in FACTORY:
        o = R[(arm, name)]['oof']
        lv = log_loss(Y[:HALF], o[:HALF], labels=[0, 1, 2])
        lh = log_loss(Y[HALF:], o[HALF:], labels=[0, 1, 2])
        print(f'  {arm:<4} {name:<5} valid {lv:.4f}  hold {lh:.4f}  차 {lv - lh:+.4f}')

### 원본(단일 체크포인트) 대비 — 같은 fold, 같은 행

`boost_oof.npz` 는 `Boost.ipynb` 가 `ABE = ckpt 단일`로 낸 OOF 다.
fold 분할이 `StratifiedKFold(5, shuffle, random_state=SEED)` 에 `Y` 만 의존하므로
**두 실행의 fold 배정이 완전히 같다** — 행 단위로 짝지어 bootstrap 을 걸 수 있다.

축 분해를 함께 찍는다. 가설이 맞다면 이득은 **승부축에서** 나와야 한다.


In [ ]:
if not BASE_OOF.exists():
    print(f'생략 — 원본 OOF 없음: {BASE_OOF}  (Boost.ipynb 를 먼저 실행)')
else:
    _b = np.load(BASE_OOF)
    assert (_b['ids'] == IDS).all() and (_b['y'] == Y).all(), '원본 OOF 행 정렬 불일치'
    print('=== 단일 -> 앙상블 (양수 = 앙상블이 이득, 문턱 0.0005) ===')
    print(f'  {"구성":<12} {"단일":>8} {"앙상블":>8} {"Δtie축":>9} {"Δ승부축":>9}   paired CI')
    for arm, name in R:
        k = f'oof__{arm}__{name}'
        if k not in _b:
            print(f'  {arm}·{name}: 원본에 없음')
            continue
        p0 = _b[k].astype(np.float64)
        p1 = R[(arm, name)]['oof']
        a0, a1 = axis_report(p0, Y), axis_report(p1, Y)
        d = paired_ci(per_row_ll(p0, Y), per_row_ll(p1, Y))
        print(f'  {arm + "·" + name:<12} {a0["총 ll"]:8.4f} {a1["총 ll"]:8.4f} '
              f'{a1["tie축"] - a0["tie축"]:+9.4f} {a1["승부축"] - a0["승부축"]:+9.4f}   ' + fmt_ci(*d))
    print('\n  Δ 축은 앙상블 − 단일이라 음수가 개선이다. 가설: 승부축만 내려가고 tie축은 제자리.')


### T 스케일링을 부스터 출력에 걸어야 하나 (평가만, 미적용)

입력 로짓에 T 를 거는 건 무의미하다 — 트리는 한 열을 상수로 나눠도 분할 순서가 안 바뀐다.
논점은 **출력** 쪽이다: swap-TTA 로 확률 두 벌을 평균하면 중앙으로 당겨져 과소확신이 생길 수 있고,
T 는 그걸 되돌리는 스칼라 하나다.

여기서는 **적용하지 않고 얼마나 벌리는지만 잰다.** T 는 두 방식으로 구한다:
- **교차적합** — fold k 의 T 를 나머지 4 fold 의 OOF 로 피팅해 fold k 에 적용. 자기적합 낙관이 없다. (판정 기준)
- 자기적합 — OOF 전체로 피팅해 그대로 적용. 스칼라 1개라 낙관은 ~1e-4 지만 참고로만 본다.

교차적합 이득이 문턱 0.0005 를 넘으면 배포 파이프라인에 T 를 넣는다.

In [ ]:
def apply_T(p, T):
    z = np.log(np.clip(p, EPS, 1.0)) / T          # 확률 -> 로짓(상수항은 softmax 가 흡수)
    z -= z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

# cv_boost 와 동일한 분할을 재현 — StratifiedKFold 는 y 와 행 수만 보므로 더미 X 로 같은 fold 가 나온다
_skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
FOLD = np.zeros(len(Y), int)
for _k, (_, _te) in enumerate(_skf.split(np.zeros((len(Y), 1)), Y)):
    FOLD[_te] = _k
assert np.bincount(FOLD).min() > 0

OOF_T = {}
print(f'{"구성":<12} {"raw ll":>8} {"T(교차)":>9} {"Δ":>8}  {"T 값":<22} {"자기적합":>9}')
for arm, name in R:
    p = R[(arm, name)]['oof']
    pT = np.zeros_like(p)
    Ts = []
    for k in range(N_SPLITS):
        tr, te = FOLD != k, FOLD == k
        T = fit_temperature(np.log(np.clip(p[tr], EPS, 1.0)), Y[tr])   # 다른 fold 로만 피팅
        pT[te] = apply_T(p[te], T)
        Ts.append(T)
    T_self = fit_temperature(np.log(np.clip(p, EPS, 1.0)), Y)
    OOF_T[(arm, name)] = pT
    ll_raw = log_loss(Y, p, labels=[0, 1, 2])
    ll_cv = log_loss(Y, pT, labels=[0, 1, 2])
    ll_self = log_loss(Y, apply_T(p, T_self), labels=[0, 1, 2])
    print(f'{arm + "·" + name:<12} {ll_raw:8.4f} {ll_cv:9.4f} {ll_raw - ll_cv:+8.4f}  '
          f'{"[" + " ".join(f"{t:.2f}" for t in Ts) + "]":<22} {ll_self:9.4f}')

print('\n=== raw -> T(교차적합) paired bootstrap (양수 = T 가 이득, 문턱 0.0005) ===')
for arm, name in R:
    d = paired_ci(per_row_ll(R[(arm, name)]['oof'], Y), per_row_ll(OOF_T[(arm, name)], Y))
    print(f'  {arm + "·" + name:<12} ' + fmt_ci(*d))
print('\n  T > 1 이면 과신(=TTA 평균에도 불구하고 날카롭다), T < 1 이면 과소확신이라 더 날카롭게 해야 한다.')

### 손실 해부 — 어디서 얼마나 깎이나

총 ll 은 평균이라 "어느 행이 얼마를 물어냈는지"를 감춘다. 다섯 갈래로 쪼갠다:

1. **클래스별 기여** — A승/B승/tie 각각이 총 손실의 몇 %를 차지하나
2. **집중도** — 상위 몇 %의 어려운 행이 손실의 몇 %를 먹나 (꼬리가 지배하면 평균 개선은 그 꼬리 싸움이다)
3. **캘리브레이션** — 신뢰도 구간별 예측 확신 vs 실제 정답률, ECE (위 T 판단의 근거)
4. **축 + 회수율** — tie축/승부축이 각각 무정보 기준 대비 몇 %를 회수했나. 오라클(축 하나를 완벽히
   맞혔을 때 남는 손실)이 곧 다른 축의 값이므로 남은 여지가 어디인지 바로 읽힌다
5. **전문가 합의 층화** — ABE 와 TB 의 tie 판정이 엇갈리는 행에서 부스터가 중재를 해내고 있나

In [ ]:
NOINFO_TIE = float(-(prior[2] * np.log(prior[2]) + (1 - prior[2]) * np.log(1 - prior[2])))
NOINFO_WIN = float(np.log(2) * (1 - prior[2]))          # 승부축은 비-tie 비율로 스케일된다

# 전문가 자체 판정 — 층화용 (부스터 입력 원본에서 직접 뽑는다)
_abe_p = np.exp(L_ABE - L_ABE.max(1, keepdims=True))
_abe_p /= _abe_p.sum(1, keepdims=True)
_tb_p = np.exp(L_TB - L_TB.max(1, keepdims=True))
_tb_p /= _tb_p.sum(1, keepdims=True)
ABE_TIE = _abe_p[:, 2] > 0.5          # ABE 가 tie 라고 보나
TB_TIE = _tb_p[:, 1] > 0.5            # TB 가 tie 라고 보나 (2-class 의 1번 열)

def dissect(p, y, tag):
    e = per_row_ll(p, y)
    tot = e.mean()
    rep = axis_report(p, y)
    print(f'\n{"=" * 78}\n[{tag}]  총 ll {tot:.4f}')

    print('\n  1) 클래스별 손실 기여')
    print(f'     {"클래스":<6} {"n":>6} {"비율":>7} {"평균손실":>9} {"기여":>8} {"기여%":>7}')
    for c, nm in enumerate(('A승', 'B승', 'tie')):
        m = y == c
        print(f'     {nm:<6} {int(m.sum()):>6,} {m.mean():7.3f} {e[m].mean():9.4f} '
              f'{e[m].sum() / len(y):8.4f} {e[m].sum() / e.sum() * 100:6.1f}%')

    print('\n  2) 손실 집중도 (어려운 행 순)')
    o = np.argsort(-e)
    for q in (0.05, 0.10, 0.20, 0.50):
        k = int(q * len(y))
        print(f'     상위 {q:>4.0%} 행이 총 손실의 {e[o[:k]].sum() / e.sum() * 100:5.1f}% '
              f'(평균손실 {e[o[:k]].mean():.3f})')

    print('\n  3) 캘리브레이션 — 예측 확신 vs 실제 정답률')
    conf, pred = p.max(1), p.argmax(1)
    hit = (pred == y).astype(float)
    b = np.clip((conf * 10).astype(int), 0, 9)
    ece = 0.0
    for i in range(10):
        m = b == i
        if m.sum() < 20:
            continue
        gap = conf[m].mean() - hit[m].mean()
        ece += m.mean() * abs(gap)
        print(f'     {i / 10:.1f}~{(i + 1) / 10:.1f}  n {int(m.sum()):>5,}  '
              f'확신 {conf[m].mean():.3f}  실제 {hit[m].mean():.3f}  차 {gap:+.3f}'
              + ('  과신' if gap > 0.02 else ('  과소확신' if gap < -0.02 else '')))
    print(f'     ECE {ece:.4f}   (전체 차 {conf.mean() - hit.mean():+.4f})')

    print('\n  4) 축 분해 + 회수율')
    print(f'     tie축  {rep["tie축"]:.4f}  (총 손실의 {rep["tie축"] / tot * 100:4.1f}%)  '
          f'무정보 {NOINFO_TIE:.4f} 대비 {(1 - rep["tie축"] / NOINFO_TIE) * 100:5.2f}% 회수')
    print(f'     승부축 {rep["승부축"]:.4f}  (총 손실의 {rep["승부축"] / tot * 100:4.1f}%)  '
          f'무정보 {NOINFO_WIN:.4f} 대비 {(1 - rep["승부축"] / NOINFO_WIN) * 100:5.2f}% 회수')
    print(f'     오라클: tie 를 완벽히 맞히면 {rep["승부축"]:.4f} 만 남고, '
          f'승부를 완벽히 맞히면 {rep["tie축"]:.4f} 가 남는다')

    print('\n  5) 전문가 tie 판정 합의 층화')
    strata = (('둘 다 tie', ABE_TIE & TB_TIE), ('둘 다 비-tie', ~ABE_TIE & ~TB_TIE),
              ('ABE만 tie', ABE_TIE & ~TB_TIE), ('TB만 tie', ~ABE_TIE & TB_TIE))
    for nm, m in strata:
        if m.sum() < 30:
            print(f'     {nm:<12} n {int(m.sum()):>6,}  (표본 부족)')
            continue
        print(f'     {nm:<12} n {int(m.sum()):>6,} ({m.mean():5.1%})  실제 tie율 {(y[m] == 2).mean():.3f}  '
              f'평균손실 {e[m].mean():.4f}  기여 {e[m].sum() / e.sum() * 100:5.1f}%')

# 최선 구성 + arm 별 3종 앙상블을 해부한다
_best = min(R, key=lambda k: R[k]['mean']['총 ll'])
dissect(R[_best]['oof'], Y, f'최선 단일: {_best[0]}·{_best[1]}')
for arm in ARMS:
    lp = np.mean([np.log(np.clip(R[(arm, n)]['oof'], EPS, 1)) for n in FACTORY], axis=0)
    pe = np.exp(lp)
    pe /= pe.sum(1, keepdims=True)
    dissect(pe, Y, f'{arm} 3종 앙상블')

### 전문가 몫 해부 — tie축은 TB 가, 승부축은 ABE 가 끌고 있나

설계 의도는 **분업**이다: 승부축은 ABE, tie축은 TB, 나머지 사각지대는 Arith.
그런데 부스터에는 로짓이 통째로 들어가서 누가 어느 축을 끌고 있는지 총점만 봐서는 안 보인다.
`총 ll` 이나 gain 중요도로는 답이 안 나온다 — gain 은 3-class 목적함수 전체에 대한 값이라
축이 섞여 있고, CatBoost 는 다출력 트리라 클래스별로 쪼개지지도 않는다.

**재매개화로 축을 분리한다.** softmax 는 상수 덧셈에 불변이므로 3-class 로짓 블록의 자유도는 2개뿐이다:

| 신호 | 정의 | 역할 |
|---|---|---|
| `s` | `z_tie − logaddexp(z_a, z_b)` = `logit(p_tie)` | **tie축을 혼자 결정** |
| `d` | `z_a − z_b` = `logit(p_a │ 비-tie)` | **승부축을 혼자 결정** |

`(z_a, z_b, z_tie) ↔ (s, d)` 는 무손실이라 성능이 보존되고(1단계에서 검산한다),
대신 **"ABE 의 tie 신호"만 콕 집어 지우는** 조작이 가능해진다. TB 는 2-class 라 `s` 하나뿐이다 —
승부축 열이 아예 없다는 게 설계 그대로다.

swap 은 이 좌표계에서 열 치환이 아니라 **부호 반전**이 된다(`s` 불변, `d → −d`). 대칭성은 동일하다.

다섯 갈래로 찍는다:

1. **무손실 검산** — 재매개 행렬로 다시 CV 해서 원본 arm 과 총 ll 이 같은지
2. **축별 permutation importance** — 신호를 하나씩 망가뜨렸을 때 tie축/승부축이 각각 얼마나 오르나 (**사용량**)
3. **삭제 재학습** — 그 신호를 아예 빼고 다시 학습했을 때의 손해 (**한계 가치**. 중복이 있으면 2번보다 작다)
4. **부스터 tie 로짓의 선형 분해** — `logit(t_부스터)` 를 세 tie 신호로 회귀한 분산 기여 몫 (합 = R²)
5. **불일치 중재** — ABE 와 TB 의 tie 판정이 갈릴 때 부스터가 누구 편을 드나


In [ ]:
DIAG_BOOST = 'cat'      # 해부에 쓸 부스터 (실측 최선). 빠르게 보려면 'lgbm'
PERM_REPS  = 3          # permutation 반복 — 셔플 우연을 줄인다

# ── 재매개화 ── 3-class 블록 -> (tie 신호 s, 승부 신호 d). 자유도 2 라 무손실이다.
def sd_of(blk):
    return blk[:, 2] - np.logaddexp(blk[:, 0], blk[:, 1]), blk[:, 0] - blk[:, 1]

_c, _m, _sg = [], [], []
_s, _d = sd_of(L_ABE)
_c += ['ABE_s(tie)', 'ABE_d(승부)']; _m += [_s, _d]; _sg += [1.0, -1.0]
_c += ['TB_s(tie)'];                 _m += [L_TB[:, 1] - L_TB[:, 0]]; _sg += [1.0]
_ari = list(ARITH_MODELS) if ARITH_MODE == 'all' else ['mean']
for _i, _nm in enumerate(_ari):
    _s, _d = sd_of(L_ARI[:, 3 * _i:3 * _i + 3])
    _c += [f'ARI_{_nm}_s(tie)', f'ARI_{_nm}_d(승부)']; _m += [_s, _d]; _sg += [1.0, -1.0]

XR   = np.column_stack(_m)
SGN  = np.array(_sg)                      # swap: s 불변, d 부호반전
IDX  = {c: i for i, c in enumerate(_c)}
COLS_R = {'AT': ['ABE_s(tie)', 'ABE_d(승부)', 'TB_s(tie)'], 'ATR': list(_c)}
GROUPS = {
    'ABE tie (s)':   ['ABE_s(tie)'],
    'ABE 승부 (d)':  ['ABE_d(승부)'],
    'TB tie (s)':    ['TB_s(tie)'],
    'Arith tie (s)':  [c for c in _c if c.startswith('ARI') and c.endswith('s(tie)')],
    'Arith 승부 (d)': [c for c in _c if c.startswith('ARI') and c.endswith('d(승부)')],
}
assert (SGN[[IDX[c] for c in _c if c.endswith('d(승부)')]] == -1).all()
assert (SGN[[IDX[c] for c in _c if c.endswith('s(tie)')]] == +1).all()

# ── 하네스 ── cv_boost 와 동일한 분할·증강이되 fold 모델을 보관한다(permutation 에 필요)
def pred_tta_sgn(m, Xp, sgn):
    p = m.predict_proba(Xp)
    if not USE_SWAP:
        return p
    return (p + m.predict_proba(Xp * sgn)[:, [1, 0, 2]]) / 2

def cv_keep(name, X, sgn, y=None, seed=SEED):
    y = Y if y is None else y
    n_inner = int(round(INNER_FRAC * len(y)))
    skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=seed)
    models, folds = [], []
    for k, (rest, te) in enumerate(skf.split(X, y)):
        tr, es = train_test_split(rest, test_size=n_inner, stratify=y[rest],
                                  random_state=seed + k, shuffle=True)
        if USE_SWAP:
            Xf = np.vstack([X[tr], X[tr] * sgn]); yf = np.hstack([y[tr], swap_y(y[tr])])
        else:
            Xf, yf = X[tr], y[tr]
        models.append(fit_one(name, FACTORY[name](), Xf, yf, X[es], y[es]))
        folds.append(te)
    return models, folds

# gidx 를 주면 그 열들을 '같은 행 셔플'로 함께 망가뜨린다 — 그룹 내 상관은 보존, 라벨과의 연결만 끊는다
def oof_of(models, folds, X, sgn, gidx=None, rng=None):
    oof = np.zeros((len(Y), 3))
    for m, te in zip(models, folds):
        Xt = X[te]
        if gidx is not None:
            Xt = Xt.copy()
            Xt[:, gidx] = Xt[rng.permutation(len(te))][:, gidx]
        oof[te] = pred_tta_sgn(m, Xt, sgn)
    return oof

print(f'해부 부스터 {DIAG_BOOST} | permutation {PERM_REPS}회 | 재매개 열 {len(_c)}개')
for arm in ARMS:
    cols = COLS_R[arm]
    ci = [IDX[c] for c in cols]
    X, sg = XR[:, ci], SGN[ci]
    sub = {k: v for k, v in GROUPS.items() if all(c in cols for c in v) and v}
    print(f'\n{"=" * 84}\n[{arm}]  재매개 열: {cols}')

    # 1) 무손실 검산 — 원본 arm 과 총 ll 이 붙어야 재매개가 옳다
    models, folds = cv_keep(DIAG_BOOST, X, sg)
    base = oof_of(models, folds, X, sg)
    rep0 = axis_report(base, Y)
    ref = R[(arm, DIAG_BOOST)]['mean']['총 ll']
    print(f'\n  1) 무손실 검산   재매개 {rep0["총 ll"]:.4f} (tie축 {rep0["tie축"]:.4f} / '
          f'승부축 {rep0["승부축"]:.4f})  vs 원본 {ref:.4f}  차 {rep0["총 ll"] - ref:+.4f}')
    print(f'     * fold 평균이 아니라 OOF 전체 점수라 0.00x 차이는 프로토콜 차이로도 난다')

    # 2) 축별 permutation importance — 사용량
    print(f'\n  2) 축별 permutation importance (양수 = 그 신호를 지우면 그만큼 나빠진다)')
    print(f'     {"신호":<16}{"Δ총":>9}{"Δtie축":>10}{"Δ승부축":>10}   해석')
    for gname, gcols in sub.items():
        gidx = [cols.index(c) for c in gcols]
        rng = np.random.default_rng(SEED)
        ds = [axis_report(oof_of(models, folds, X, sg, gidx, rng), Y) for _ in range(PERM_REPS)]
        dt = np.mean([r['총 ll'] for r in ds]) - rep0['총 ll']
        di = np.mean([r['tie축'] for r in ds]) - rep0['tie축']
        dw = np.mean([r['승부축'] for r in ds]) - rep0['승부축']
        who = 'tie축' if di > 3 * abs(dw) else ('승부축' if dw > 3 * abs(di) else '양축')
        print(f'     {gname:<16}{dt:+9.4f}{di:+10.4f}{dw:+10.4f}   {who} 담당')

    # 3) 삭제 재학습 — 한계 가치 (중복이 있으면 2번보다 작게 나온다)
    print(f'\n  3) 삭제 재학습 (양수 = 그 신호를 빼면 그만큼 나빠진다 = 대체 불가능한 몫)')
    print(f'     {"제거":<16}{"Δ총":>9}{"Δtie축":>10}{"Δ승부축":>10}')
    for gname, gcols in sub.items():
        keep = [i for i, c in enumerate(cols) if c not in gcols]
        if not keep:
            continue
        mk, fk = cv_keep(DIAG_BOOST, X[:, keep], sg[keep])
        rk = axis_report(oof_of(mk, fk, X[:, keep], sg[keep]), Y)
        print(f'     {gname:<16}{rk["총 ll"] - rep0["총 ll"]:+9.4f}'
              f'{rk["tie축"] - rep0["tie축"]:+10.4f}{rk["승부축"] - rep0["승부축"]:+10.4f}')

    # 4) 부스터 tie 로짓의 선형 분해 — 분산 기여 몫 (합 = R^2)
    tie_cols = [c for c in cols if c.endswith('s(tie)')]
    t = np.clip(base[:, 2], EPS, 1 - EPS)
    yt = np.log(t / (1 - t))
    Z = np.column_stack([XR[:, IDX[c]] for c in tie_cols])
    Z = (Z - Z.mean(0)) / Z.std(0)
    yz = (yt - yt.mean()) / yt.std()
    beta = np.linalg.lstsq(np.c_[np.ones(len(Z)), Z], yz, rcond=None)[0][1:]
    contrib = beta * np.array([np.corrcoef(Z[:, j], yz)[0, 1] for j in range(Z.shape[1])])
    r2 = float(contrib.sum())
    print(f'\n  4) 부스터 tie 로짓 = 세 tie 신호의 선형 결합? (R² {r2:.4f})')
    print(f'     {"tie 신호":<18}{"표준화 β":>10}{"분산 기여":>10}{"몫%":>8}')
    for c, b, k in zip(tie_cols, beta, contrib):
        print(f'     {c:<18}{b:+10.3f}{k:+10.4f}{k / r2 * 100:7.1f}%')

    # 5) 불일치 중재 — ABE 와 TB 의 tie 판정이 갈릴 때 부스터는 누구를 따르나
    _pa = np.exp(L_ABE - L_ABE.max(1, keepdims=True)); _pa /= _pa.sum(1, keepdims=True)
    _pt = np.exp(L_TB - L_TB.max(1, keepdims=True)); _pt /= _pt.sum(1, keepdims=True)
    a_tie, t_tie = _pa[:, 2], _pt[:, 1]
    print(f'\n  5) tie 판정 불일치 시 부스터의 선택 (부스터 p_tie 가 어느 쪽에 붙나)')
    print(f'     {"층":<14}{"n":>6}{"실제tie율":>10}{"ABE":>8}{"TB":>8}{"부스터":>8}   추종')
    for nm, m in (('둘 다 tie', (a_tie > .5) & (t_tie > .5)),
                  ('둘 다 비-tie', (a_tie <= .5) & (t_tie <= .5)),
                  ('ABE만 tie', (a_tie > .5) & (t_tie <= .5)),
                  ('TB만 tie', (a_tie <= .5) & (t_tie > .5))):
        if m.sum() < 30:
            print(f'     {nm:<14}{int(m.sum()):>6}  (표본 부족)'); continue
        act, bp = (Y[m] == 2).mean(), base[m, 2].mean()
        follow = ''
        if 'ABE만' in nm or 'TB만' in nm:
            follow = 'ABE' if abs(bp - a_tie[m].mean()) < abs(bp - t_tie[m].mean()) else 'TB'
            follow += f' (실제 {act:.3f} 에 더 가까운 쪽: ' + \
                      ('ABE' if abs(act - a_tie[m].mean()) < abs(act - t_tie[m].mean()) else 'TB') + ')'
        print(f'     {nm:<14}{int(m.sum()):>6}{act:>10.3f}{a_tie[m].mean():>8.3f}'
              f'{t_tie[m].mean():>8.3f}{bp:>8.3f}   {follow}')

print(f'\n{"=" * 84}')
print('읽는 법 — 설계대로면: (a) TB tie(s) 는 tie축에만 큰 Δ, 승부축 Δ ≈ 0')
print('           (b) ABE 승부(d) 는 승부축에만 큰 Δ')
print('           (c) 4) 에서 TB 의 몫이 ABE tie 보다 커야 "tie 는 TB 담당" 이 성립')
print('           (d) 3) 이 2) 보다 훨씬 작으면 그 신호는 다른 신호로 대체 가능하다는 뜻(중복)')


### 계층 구조 검증 — tie 판정과 승부 판정을 아키텍처로 분리하면

앞 셀에서 분업이 실측으로 확인됐다(TB→tie축, ABE `d`→승부축). 그러면 그 분업을 **구조로 강제**하면
어떤가: TB 가 `t` 를 정하고, 비-tie 몫을 ABE+Arith 가 가른다.

$$p = [\,(1-t)\cdot r,\ \ (1-t)\cdot(1-r),\ \ t\,]$$

**분해 자체는 손실이 0 이다.** `총 ll = tie축 + 승부축` 이 임의의 확률벡터에 성립하는 항등식이고,
tie축은 `t` 에만, 승부축은 `r` 에만 의존한다 — 두 축은 원리적으로 따로 최적화된다.
그러니 쟁점은 분해가 아니라 **각 헤드가 무엇을 보고, 무엇으로 학습하느냐** 셋뿐이다.

**구조로 얻는 것 (joint 부스터가 못 하는 것)**

1. **대칭성을 증강이 아니라 구조로 얻는다.** tie 헤드 입력을 swap 불변량(`s`, `|d|`)으로만 채우면
   그 헤드는 **정의상 swap 대칭**이라 증강·TTA 가 아예 불필요하다. 승부 헤드는 입력이 홀함수(`d`)라
   `(−d, 1−y)` 증강으로 **반대칭을 정확히** 강제할 수 있다. 지금 joint 부스터는 이걸 데이터 증강으로
   근사할 뿐이라 절대 정확히 만족하지 못한다.
2. **승부 헤드의 학습 표본이 추정 대상과 일치한다.** 승부축은 `r = P(A승 │ 비-tie)` 인데
   지금은 tie 행까지 섞인 3-class 목적함수로 적합한다. 비-tie 행만으로 이진 학습하면
   목적함수 = 지표가 된다(평가도 비-tie 행에서만 하므로 공변량 이동이 없다).
3. **용량이 축마다 온전히 쓰인다.** 지금은 트리 하나가 두 축을 겸업한다.

**구조로 잃을 수 있는 것 — 축간 상호작용.** "TB 가 tie 라고 볼수록 ABE 마진을 덜 믿어라" 같은 규칙은
계층화하면 표현할 수 없다. 앞 셀 permutation 에서 `TB_s → 승부축` Δ 가 ±0.0002 로 잡혔으니 작을
것으로 보이지만, 여기서 직접 잰다.

**그리고 함정 하나 — tie 헤드를 TB 만으로 좁히면 안 될 수 있다.**
앞 셀에서 `ABE_s` 삭제가 공짜였던 이유가 "TB 가 대체해서"가 아니라 **`|ABE_d|`(마진 절대값)가 tie 를
예측해서**일 수 있다. 마진이 좁으면 tie 다. 별도 진단의 `TB만 0.5782 vs ABE+TB 0.5753` 이 그 크기
(+0.0029)를 시사한다. 그래서 **입력 폭을 3단계로 나눠** 재야 "분해 이득"과 "입력 제한 손실"이 분리된다.

| 코드 | tie 헤드 입력 | 승부 헤드 입력 | 검증하는 것 |
|---|---|---|---|
| `J` | — (현행 joint 3-class, **원좌표** 전 열) | — | 현행 기준선 |
| `J-rep` | — (joint 3-class, **재매개 좌표**) | — | **공정 기준선** (좌표 효과 제거) |
| `F-full` | 모든 `s` + 모든 `|d|` | 모든 `d` + 모든 `s` | 순수 **분해 이득** (정보 제한 없음) |
| `F-mid` | `TB_s` + 모든 `|d|` | 모든 `d` | 설계 의도 + 마진 보존 |
| `F-tb` | **`TB_s` 만** | 모든 `d` | 질문 그대로의 엄격 계층화 |

판정은 `J-rep` 기준으로 한다 — `J` 와 재면 재매개 비용(트리 분할 기하 변화, 정보 손실 아님)이
분해 비용에 섞여 들어간다. `F-full − J-rep` 가 양수면 분해 자체가 이득이고, `F-tb − F-mid` 가 곧 **마진 정보를 버리는 값** 이다.
fold 는 `FOLD` 를 그대로 써서 `J` 와 행 단위로 짝지어 bootstrap 한다.


In [ ]:
# 앞 셀(재매개화)의 XR/IDX/SGN/_c 를 그대로 쓴다. FOLD 는 T 셀에서 만든 것과 동일 분할.
if 'FOLD' not in dir():
    _sk = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
    FOLD = np.zeros(len(Y), int)
    for _k, (_, _te) in enumerate(_sk.split(np.zeros((len(Y), 1)), Y)):
        FOLD[_te] = _k

# 이진 헤드 — 3-class 부스터와 같은 하이퍼파라미터 계열
def make_bin(name):
    if name == 'lgbm':
        return lgb.LGBMClassifier(objective='binary', learning_rate=0.03, n_estimators=3000,
                                  num_leaves=15, min_child_samples=60, colsample_bytree=0.9,
                                  subsample=0.9, subsample_freq=1, reg_lambda=1.0,
                                  random_state=SEED, n_jobs=-1, verbosity=-1)
    if name == 'xgb':
        return xgb.XGBClassifier(objective='binary:logistic', learning_rate=0.03,
                                 n_estimators=3000, max_depth=4, min_child_weight=20,
                                 subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
                                 tree_method='hist', eval_metric='logloss',
                                 early_stopping_rounds=100, random_state=SEED,
                                 n_jobs=-1, verbosity=0)
    return cb.CatBoostClassifier(loss_function='Logloss', eval_metric='Logloss',
                                 iterations=3000, learning_rate=0.03, depth=4, l2_leaf_reg=6.0,
                                 early_stopping_rounds=100, bootstrap_type='Bernoulli',
                                 subsample=0.9, border_count=254, random_seed=SEED,
                                 verbose=0, allow_writing_files=False)

S_ALL = [c for c in _c if c.endswith('s(tie)')]
D_ALL = [c for c in _c if c.endswith('d(승부)')]

# 열 이름 -> (값, swap 부호). |d| 는 swap 불변이라 부호 +1.
def col_of(nm):
    if nm.startswith('|'):
        return np.abs(XR[:, IDX[nm[1:-1]]]), +1.0
    return XR[:, IDX[nm]], SGN[IDX[nm]]

def build(names):
    if not names:
        return np.zeros((len(Y), 0)), np.zeros(0)
    vs, sg = zip(*(col_of(n) for n in names))
    return np.column_stack(vs), np.array(sg)

# tie 헤드(이진, 전체 행) + 승부 헤드(이진, 비-tie 행) -> 3-class 조립
def cv_cascade(name, tie_names, win_names, seed=SEED):
    XT, sgT = build(tie_names)
    XW, sgW = build(win_names)
    # 계약: tie 헤드 입력은 전부 swap 불변이어야 구조적 대칭이 성립한다
    assert (sgT == 1).all(), f'tie 헤드에 swap 홀함수 열이 있다: {tie_names}'
    assert (sgW == -1).any(), f'승부 헤드에 방향 열이 없다: {win_names}'

    yt = (Y == 2).astype(int)
    yw = (Y == 0).astype(int)
    t_out, r_out, iters = np.zeros(len(Y)), np.zeros(len(Y)), []
    for k in range(N_SPLITS):
        tei = np.where(FOLD == k)[0]
        rest = np.where(FOLD != k)[0]

        # ── tie 헤드 ── 입력이 swap 불변이라 증강·TTA 불필요 (구조적 대칭)
        n_in = int(round(INNER_FRAC * len(Y)))
        tr, es = train_test_split(rest, test_size=n_in, stratify=yt[rest],
                                  random_state=seed + k, shuffle=True)
        m1 = fit_one(name, make_bin(name), XT[tr], yt[tr], XT[es], yt[es])
        t_out[tei] = m1.predict_proba(XT[tei])[:, 1]

        # ── 승부 헤드 ── 비-tie 행만(추정 대상 = 지표). (x*sg, 1-y) 증강으로 반대칭 강제
        nz = rest[Y[rest] != 2]
        n_in2 = max(50, int(round(INNER_FRAC * len(nz))))
        tr2, es2 = train_test_split(nz, test_size=n_in2, stratify=yw[nz],
                                    random_state=seed + k, shuffle=True)
        Xf = np.vstack([XW[tr2], XW[tr2] * sgW])
        yf = np.hstack([yw[tr2], 1 - yw[tr2]])
        m2 = fit_one(name, make_bin(name), Xf, yf, XW[es2], yw[es2])
        r_out[tei] = (m2.predict_proba(XW[tei])[:, 1]
                      + 1 - m2.predict_proba(XW[tei] * sgW)[:, 1]) / 2
        iters.append((best_iter(name, m1), best_iter(name, m2)))

    p = np.column_stack([(1 - t_out) * r_out, (1 - t_out) * (1 - r_out), t_out])
    return p / p.sum(1, keepdims=True), iters

CAS_BOOST = DIAG_BOOST      # 앞 셀과 같은 부스터로 J 와 직접 비교
print(f'계층 구조 검증 | 부스터 {CAS_BOOST} | fold {N_SPLITS} (J 와 동일 분할)\n')

CASCADE = {}
for arm in ARMS:
    cols = COLS_R[arm]
    s_in = [c for c in S_ALL if c in cols]
    d_in = [c for c in D_ALL if c in cols]
    abs_in = [f'|{c}|' for c in d_in]
    variants = {
        'F-full': (s_in + abs_in, d_in + s_in),
        'F-mid':  (['TB_s(tie)'] + abs_in, d_in),
        'F-tb':   (['TB_s(tie)'], d_in),
    }
    pJ = R[(arm, CAS_BOOST)]['oof']
    rJ = axis_report(pJ, Y)
    # 공정 기준선 J-rep — 계층화와 '같은 재매개 좌표'에서 학습한 joint 3-class.
    # J 와 직접 비교하면 재매개 비용(트리 분할 기하 변화)과 분해 비용이 섞여 판정이 오염된다.
    _ci = [IDX[c] for c in cols]
    _mk, _fk = cv_keep(CAS_BOOST, XR[:, _ci], SGN[_ci])
    pJR = oof_of(_mk, _fk, XR[:, _ci], SGN[_ci])
    CASCADE[(arm, 'J-rep')] = pJR
    rJR = axis_report(pJR, Y)
    print(f'{"=" * 96}\n[{arm}]  J (원좌표 joint) 총 {rJ["총 ll"]:.4f} '
          f'(tie {rJ["tie축"]:.4f} / 승부 {rJ["승부축"]:.4f})   '
          f'J-rep (재매개 joint) 총 {rJR["총 ll"]:.4f} '
          f'(tie {rJR["tie축"]:.4f} / 승부 {rJR["승부축"]:.4f})')
    print(f'  재매개 비용 J -> J-rep {rJR["총 ll"] - rJ["총 ll"]:+.4f}  (분해와 무관한 좌표 효과)')
    print(f'  {"구조":<8}{"총 ll":>9}{"tie축":>9}{"승부축":>9}{"Δ총":>9}{"Δtie":>9}{"Δ승부":>9}   '
          f'vs J-rep (양수 = 계층화 이득)')
    for vname, (tn, wn) in variants.items():
        p, it = cv_cascade(CAS_BOOST, tn, wn)
        CASCADE[(arm, vname)] = p
        r = axis_report(p, Y)
        d = paired_ci(per_row_ll(pJR, Y), per_row_ll(p, Y))
        print(f'  {vname:<8}{r["총 ll"]:9.4f}{r["tie축"]:9.4f}{r["승부축"]:9.4f}'
              f'{r["총 ll"] - rJR["총 ll"]:+9.4f}{r["tie축"] - rJR["tie축"]:+9.4f}'
              f'{r["승부축"] - rJR["승부축"]:+9.4f}   ' + fmt_ci(*d))
    # 입력 제한의 값 — 같은 구조끼리라 순수한 정보 손실만 남는다
    for a, b in (('F-full', 'F-mid'), ('F-mid', 'F-tb')):
        d = paired_ci(per_row_ll(CASCADE[(arm, a)], Y), per_row_ll(CASCADE[(arm, b)], Y))
        print(f'  {a} -> {b:<8} (양수 = 좁히는 게 이득) ' + fmt_ci(*d))

print(f'\n{"=" * 90}')
print('읽는 법')
print('  · F-full − J-rep: 순수 분해 이득. 같은 좌표·같은 정보라 구조(대칭 강제 + 비-tie 전용')
print('                    승부 학습 + 축별 용량)만의 값이다. 음수면 축간 상호작용이 실재한다는 뜻.')
print('  · F-mid − F-tb: |d|(마진)를 tie 헤드에서 빼는 값. 앞 셀의 ABE_s 삭제가 공짜였던 게')
print('                  TB 대체 때문인지 마진 대체 때문인지를 여기서 가른다.')
print('  · 채택 시 이득: tie 헤드는 구조적 대칭이라 swap 증강·TTA 가 불필요해져 추론이 절반이 된다.')


### 배포용 최종 적합 + 저장

CV 는 측정 전용이었다. 배포 모델은 **stage2 전량**으로 다시 적합한다 —
ES 는 같은 비율(10%)을 한 번 떼어 쓰고, 그 결과 `best_iteration` 이 CV 의 중앙값과 크게 어긋나면
데이터량 변화 이상의 무언가가 있다는 뜻이라 경고를 찍는다.

In [ ]:
FINAL = {}
tr_all, es_all = train_test_split(np.arange(len(Y)), test_size=int(round(INNER_FRAC * len(Y))),
                                  stratify=Y, random_state=SEED, shuffle=True)
for arm, (X, cols) in ARMS.items():
    for name in FACTORY:
        if USE_SWAP:
            Xf = np.vstack([X[tr_all], X[tr_all][:, PERM[arm]]])
            yf = np.hstack([Y[tr_all], swap_y(Y[tr_all])])
        else:
            Xf, yf = X[tr_all], Y[tr_all]
        m = fit_one(name, FACTORY[name](), Xf, yf, X[es_all], Y[es_all])
        FINAL[(arm, name)] = m
        bi, med = best_iter(name, m), int(np.median(R[(arm, name)]['iters']))
        flag = '  경고: CV 중앙값과 2배 이상 차이' if bi and (bi > 2 * med or bi * 2 < med) else ''
        print(f'{arm:<4} {name:<5} best_iter {bi:>5} (CV 중앙값 {med:>5}){flag}')

for (arm, name), m in FINAL.items():
    d = MODEL_DIR / arm
    os.makedirs(d, exist_ok=True)
    if name == 'lgbm':
        m.booster_.save_model(str(d / 'lgbm.txt'))
    elif name == 'xgb':
        m.save_model(str(d / 'xgb.json'))
    else:
        m.save_model(str(d / 'cat.cbm'))

meta = {
    'created': time.strftime('%Y-%m-%d %H:%M'),
    'variant': 'abe_ckpt_ensemble',
    'stage2_rows': int(len(IDS)), 'n_splits': N_SPLITS, 'inner_frac': INNER_FRAC,
    'use_swap': USE_SWAP, 'arith_mode': ARITH_MODE, 'ckpt_select': CKPT_SELECT,
    'ensemble': {'abe': ABE_ENSEMBLE, 'tb': TB_ENSEMBLE},
    'experts': {
        # steps 가 배포 계약이다 — 제출 추론도 같은 체크포인트를 같은 수만큼 평균해야 한다
        'abe': {'steps': ABE_STEPS, 'T': ABE_T, 'n_class': 3, 'root': str(ABE_ROOT),
                'note': '로짓 공간 평균 후 T 재피팅. 확률 평균 아님'},
        'tb': {'steps': TB_STEPS, 'T': TB_T, 'n_class': 2, 'root': str(TB_ROOT),
               'note': 'swap 불변 — [1,0,2] 재정렬 금지. 빈 응답은 배포본 AttnPool 의 '
                       'nan_to_num 가드로 0 벡터 처리'},
        'arith': {'models': list(ARITH_MODELS), 'npz': str(ARITH_NPZ)}},
    'columns': {a: c for a, (X, c) in ARMS.items()},
    'swap_perm': {a: PERM[a].tolist() for a in ARMS},
    'cv_mean': {f'{a}·{n}': {k: float(v) for k, v in R[(a, n)]['mean'].items()} for a, n in R},
    'cv_std_ll': {f'{a}·{n}': float(R[(a, n)]['std']['총 ll']) for a, n in R},
    'best_iters_cv': {f'{a}·{n}': R[(a, n)]['iters'] for a, n in R},
}
with open(MODEL_DIR / 'boost_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

np.savez_compressed(OOF_PATH, ids=IDS, y=Y,
                    **{f'oof__{a}__{n}': R[(a, n)]['oof'].astype(np.float32) for a, n in R})
print(f'\nsaved: {MODEL_DIR}/{{AT,ATR}}/{{lgbm.txt,xgb.json,cat.cbm}}')
print(f'saved: {MODEL_DIR}/boost_meta.json  |  {OOF_PATH}')


## 읽는 법 / 다음 단계

1. **원본 대비 셀이 이 노트북의 본론이다.** 같은 fold·같은 행이라 paired bootstrap 이 성립하고,
   Δ 가 문턱 0.0005 를 넘으면 ABE 체크포인트 앙상블을 배포에 넣는다.
   그때 **축 분해를 같이 본다** — 이득이 승부축에서 나와야 가설대로다(tie 축은 TB 담당).
2. **arm 비교**(`AT -> ATR`)는 원본과 같은 판정이다. 앙상블로 ABE 가 세지면 Arith 의 몫이
   줄어들 수 있다 — 승부축에서 겹치기 때문이다. 여기서 노이즈로 떨어지면 제출 파이프라인에서
   피쳐 추출·트리 추론을 통째로 뺄 수 있다.
3. **fold 표준편차를 함께 본다.** 평균 차이가 std 보다 작으면 fold 우연으로 뒤집힐 수 있는 차이다.
4. **valid/hold 반쪽 차이**가 크면 1차 모델 선택 편향이 stage2 로 새어든 신호다(§1.6 보조 진단).
   단 원본 실행에서 이미 valid 가 hold 보다 나빴다(+0.007~0.012) — 방향이 낙관이 아니라 비관이라
   선택 편향이 아니라 분할 자체의 난이도 차이로 읽는 게 맞다.
5. 배포 시 추론 순서: ABE(**체크포인트 N벌** × 2뷰 TTA) → TB(2뷰 TTA) → Arith(피쳐 → 트리 3종 TTA)
   → 이 부스터. **열 순서는 `boost_meta.json` 의 `columns` 가 계약이고, ABE 체크포인트 목록은
   `experts.abe.steps` 가 계약이다.** 둘 중 하나만 어긋나도 조용히 틀린 답을 낸다.
   추론 비용이 체크포인트 수만큼 곱해지므로, 이득 대비 시간 예산을 §1.6 제출 한도와 대조할 것.
